In [34]:
import pandas as pd
import numpy as np

import string

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.tokenize import sent_tokenize

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix


from gensim.models import Word2Vec
from gensim.utils import simple_preprocess

from tqdm import tqdm

In [19]:
df = pd.read_csv("IMDB Dataset.csv")

df['review'] = df['review'].str.lower()

In [20]:
def remove_html(data):
    if isinstance(data,str):
        import html
        import re 
        data = html.unescape(data)
        data  = re.sub(r'<[^>]+>', ' ',data)
        data  = re.sub(r'\s+', ' ',data)
        return data.strip()
    return data

df['review'] = df['review'].apply(remove_html)

In [21]:
punc = string.punctuation

def remove_pun(data):
    if isinstance(data, str):
        return ''.join(char for char in data if char not in punc)
    return data

df['review'] = df['review'].apply(remove_pun)

In [22]:
df.duplicated().sum()

df.drop_duplicates(inplace=True)

df.duplicated().sum()

0

In [23]:
df['sentiment'].unique()

<StringArray>
['positive', 'negative']
Length: 2, dtype: str

In [24]:
stop_words = set(stopwords.words('english'))

def tokenize_text(text):
    if isinstance(text, str) and text.strip():
        return word_tokenize(text.lower())
    return []

df['tokens'] = df['review'].apply(tokenize_text)

In [25]:
def remove_stop_words(tokens):
    if isinstance(tokens,list):
        return [word for word in tokens if word.lower() not in stop_words]
    return []

df['stop_words_rem'] = df['tokens'].apply(remove_stop_words)

In [26]:
df.columns.to_list()

['review', 'sentiment', 'tokens', 'stop_words_rem']

In [27]:
df['clean_text'] = df['stop_words_rem'].apply(lambda x: ' '.join(x) if x else '')

In [28]:
story = []
for doc in df['clean_text'] :
    raw_sent = sent_tokenize(doc)
    for sent in raw_sent:
        story.append(simple_preprocess(sent))

In [29]:
model = Word2Vec(window=10,workers=5,min_count=2)

In [30]:
model.build_vocab(story)

In [31]:
model.train(story,total_examples=model.corpus_count, epochs=model.epochs)

(27964662, 29446340)

In [32]:
def document_vector(doc):
    doc = [word for word in doc.split() if word in model.wv.index_to_key]
    return np.mean(model.wv[doc],axis=0)

In [33]:
document_vector(df['clean_text'].values[0])

array([-0.15017158, -0.00973297, -0.4596382 , -0.58654296,  0.05802993,
       -0.35623598,  0.5687917 , -0.05328573, -0.2872862 , -0.08440713,
       -0.33509278,  0.0715591 ,  0.03093891, -0.08494835, -0.10809833,
       -0.27915296,  0.14404602, -0.11566224, -0.1458765 , -0.26148814,
        0.46512818,  0.3723084 , -0.10041016, -0.03332978, -0.07915694,
       -0.00458546, -0.23088299, -0.09466131, -0.46037975, -0.3631408 ,
        0.9700441 ,  0.60388535, -0.06688966, -0.23394111, -0.17387344,
        0.41662663, -0.55296594, -0.25880536, -0.36138573, -0.21227592,
        0.21919952,  0.08131836, -0.18679754, -0.12083049,  0.20619962,
       -0.12167145, -0.01501074, -0.252551  , -0.02649415,  0.11903018,
        0.2630565 , -0.20634273, -0.0062878 , -0.13718575,  0.18053167,
        0.18038316,  0.6916473 ,  0.36342838,  0.4435826 ,  0.35527962,
       -0.36092517,  0.11544286,  0.39768323,  0.24557102, -0.12479423,
        0.19196808, -0.03825745,  0.01035381,  0.03450258,  0.01

In [35]:
X = []
for doc in tqdm(df['clean_text'].values):
    X.append(document_vector(doc))


 40%|████      | 19881/49577 [10:52<16:14, 30.47it/s]


KeyboardInterrupt: 

In [37]:
encode = LabelEncoder()
y = encode.fit_transform(df['sentiment'])


In [38]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

ValueError: Found input variables with inconsistent numbers of samples: [19881, 49577]

In [ ]:
rn = RandomForestClassifier()
rn.fit(X_train, y_train)
y_pred = rn.predict(X_test)

In [ ]:
print(f"Accuracy Score : {accuracy_score(y_test,y_pred)}")